# PolyWhisper v9 — Per-Language Augmentation (eulogikdevelopers)

hi/ta/te: NO augmentations (enough data; SpecAugment caused token-loop degeneration)
bn/mr: full SpecAugment + speed perturb (bn -14%, mr -51% in v8)

Account-independent: backbone `openai/whisper-small` downloads from HF at runtime (pinned revision). No /kaggle/input dataset dependency. 2-GPU orchestrator when 2 GPUs present, sequential fallback on 1 GPU.

In [ ]:
!pip install -q transformers datasets accelerate peft torch torchaudio evaluate jiwer sentencepiece huggingface_hub soundfile librosa resampy

In [ ]:
import resampy, librosa, soundfile, torch, transformers, datasets, peft, accelerate
print('deps OK: resampy', resampy.__version__,
      '| torch', torch.__version__,
      '| transformers', transformers.__version__)
import os, datetime
def beat(phase):
    try:
        tok = os.environ.get('HF_TOKEN', '')
        if not tok:
            print(f'[beat] {phase} (no token, local only)', flush=True)
            return
        from huggingface_hub import HfApi
        msg = f"{phase} @ {datetime.datetime.now(datetime.timezone.utc).isoformat()}"
        open('/tmp/phase.txt', 'w').write(msg + '\n')
        HfApi(token=tok).upload_file('/tmp/phase.txt', 'polywhisper_v9_kernel_progress.txt',
            repo_id='eulogik/polywhisper', repo_type='model',
            commit_message=f'v9 progress: {phase}')
        print('[beat]', msg, flush=True)
    except Exception as e:
        print('[beat] failed:', str(e)[:150], flush=True)
beat('v4-pip-done')


In [ ]:
import os, shutil
for d in ['scripts', 'polywhisper_output_gpu0', 'polywhisper_output_gpu1']:
    if os.path.exists(d): shutil.rmtree(d)
os.makedirs('scripts', exist_ok=True)
print('cleaned')

In [ ]:
import shutil
from huggingface_hub import hf_hub_download
for s in ['train_v3.py', 'eval_lang_pure.py', 'kaggle_train_resumable.py', 'normalize_ortho.py']:
    p = hf_hub_download('eulogik/polywhisper', s, repo_type='model')
    shutil.copy(p, f'scripts/{s}')
    print(' ', s)
t = open('scripts/train_v3.py').read()
assert 'AUGMENT_LANGS' in t, 'STALE train_v3.py on HF — aborting'
o = open('scripts/kaggle_train_resumable.py').read()
assert '"--augment-langs"' in o and '"bn,mr"' in o, 'STALE orchestrator on HF — aborting'
print('version check OK: per-language augmentation present')
beat('v4-scripts-ok')


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
        print('HF_TOKEN secret attached: yes (checkpoints will persist to HF)')
    else:
        print('HF_TOKEN secret empty: HF upload skipped, pull outputs via kernels output API')
except Exception as e:
    print('HF_TOKEN secret not attached — HF upload skipped, pull outputs via kernels output API:', str(e)[:120])
import torch
n = torch.cuda.device_count()
print('CUDA GPUs:', n, [torch.cuda.get_device_name(i) for i in range(n)] if n else [])


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
os.chdir('scripts')
import torch
n = torch.cuda.device_count()
print('GPUs:', n, flush=True)
beat(f'v4-training-start gpus={n}')
if n >= 2:
    print('2+ GPUs: launching 2-GPU orchestrator (train + eval + HF upload)', flush=True)
    r = subprocess.run([sys.executable, 'kaggle_train_resumable.py'])
    print('orchestrator rc=', r.returncode, flush=True)
else:
    print('single GPU: sequential fallback (all 5 langs, per-lang augment inside train_v3)',
          flush=True)
    sys.path.insert(0, '.')
    import kaggle_train_resumable as K
    SD = 'polywhisper_output_gpu0'
    if os.environ.get('HF_TOKEN'):
        K.api.token = os.environ['HF_TOKEN']
        K.fetch_remote(SD)
    r = subprocess.run([sys.executable, 'train_v3.py', '--langs', 'hi,ta,te,bn,mr',
        '--model-size', 'small', '--epochs', '5', '--batch-size', '4',
        '--encoder-lora', '--tag', '_prod', '--save-dir', SD,
        '--max-runtime-hours', '11', '--wer-eval-every', '500',
        '--augment-langs', 'bn,mr'])
    print('train rc=', r.returncode, flush=True)
    for p in Path(SD, 'data').glob('audio_indicvoices_*'):
        shutil.rmtree(p, ignore_errors=True)
    print('freed re-downloadable indicvoices FLAC (kept fleurs test audio for eval)',
          flush=True)
    for lang in ['hi', 'ta', 'te', 'bn', 'mr']:
        try:
            K.eval_lang(lang, SD)
        except Exception as e:
            print(f'eval {lang} FAILED: {e}', flush=True)
    try:
        K.upload_tree(SD, K.remote_snapshot(), force=True)
    except Exception as e:
        print('final upload failed:', str(e)[:200], flush=True)
print('MAIN DONE', flush=True)
beat('v4-main-done')


In [ ]:
from pathlib import Path
print('cwd check + output manifest:')
for d in ['polywhisper_output_gpu0', 'polywhisper_output_gpu1']:
    p = Path(d)
    if not p.exists():
        print(f'  {d}: MISSING')
        continue
    ad = p / 'adapters_v3'
    pts = sorted(x.name for x in ad.glob('*.pt')) if ad.exists() else []
    evs = sorted(x.name for x in p.glob('eval_*.json'))
    print(f'  {d}: adapters={pts}')
    print(f'  {d}: evals={evs}')
